<a href="https://colab.research.google.com/github/sjkim-audio/Bass-separator/blob/main/notebooks/transcription/11_parameter_tunning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================
# [Bass Separator] Ultimate Environment Setup (v4.0 - Best Practice)
# ==================================================================
import os
import sys
import subprocess
from google.colab import drive

print("🚀 Bass Separator 통합 환경 설정을 시작합니다...")

# -----------------------------------------------------------------
# 1. Google Drive 마운트
# -----------------------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# -----------------------------------------------------------------
# 2. GitHub 최신화 및 경로 설정 (충돌 없는 강제 동기화)
# -----------------------------------------------------------------
PROJECT_NAME = "Bass-separator"
REPO_URL = "https://github.com/sjkim-audio/Bass-separator.git"
PROJECT_PATH = f"/content/{PROJECT_NAME}"

try:
    if not os.path.exists(PROJECT_PATH):
        print(f"📦 레포지토리 클론 중... ({PROJECT_NAME})")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    else:
        print(f"🔄 레포지토리 최신화 중... (Git Fetch & Reset)")
        subprocess.run(["git", "fetch", "--all"], cwd=PROJECT_PATH, check=True)
        subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=PROJECT_PATH, check=True)
except subprocess.CalledProcessError as e:
    raise RuntimeError(f"❌ Git 동기화 실패: {e}")

# 핵심: 작업 디렉토리를 프로젝트 루트로 완벽히 고정하여 requirements.txt를 찾게 함
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
os.chdir(PROJECT_PATH)

# -----------------------------------------------------------------
# 3. 커스텀 모듈(src) 실행을 통한 의존성 설치 및 데이터셋 로드
# -----------------------------------------------------------------
try:
    from src.env_setup import init_colab_env
    from src.utils import load_data_from_drive

    # env_setup.py 내부의 함수를 호출하여 requirements.txt 기반 설치 실행
    # (이 단계에서 torchcrepe, pretty_midi 등이 정상 설치됩니다)
    init_colab_env()

    # 데이터셋 복사
    MY_DRIVE_DATA_PATH = "/content/drive/MyDrive/Bass_separator/dataset"
    load_data_from_drive(MY_DRIVE_DATA_PATH, force_update=False)

except ImportError as e:
    print(f"⚠️ 커스텀 모듈 임포트 에러: {e}")
    print("   (src/utils.py의 'import shutil' 오타가 수정되었는지 확인하세요!)")
except Exception as e:
    print(f"❌ 셋업 중단: {e}")

# -----------------------------------------------------------------
# 4. 글로벌 라이브러리 사전 적재
# -----------------------------------------------------------------
import librosa
import numpy as np
import pandas as pd
import torchcrepe
import pretty_midi

print("\n🎉 Ready to Rock! 모든 셋업과 모듈 로드가 완벽히 끝났습니다.")

🚀 Bass Separator 통합 환경 설정을 시작합니다...
Mounted at /content/drive
📦 레포지토리 클론 중... (Bass-separator)
🚀 환경 설정을 시작합니다...

🔧 [시스템] 필수 도구 확인 중...
✅ FFmpeg가 이미 설치되어 있습니다.

🐍 [파이썬] 라이브러리 확인 중...
📄 requirements.txt 파일을 발견했습니다. 의존성 패키지를 설치합니다...
📦 패키지 일괄 설치 진행 중...
✅ 패키지 일괄 설치 완료.

🏥 설치 무결성 점검 (Health Check)...
✅ 필수 라이브러리가 모두 정상적으로 준비되었습니다!

🎉 모든 환경 설정이 완료되었습니다!
🚀 데이터 복사 시작...
   📂 Source: /content/drive/MyDrive/Bass_separator/dataset
   📂 Dest  : ./dataset
🎉 데이터 준비 완료! (총 5개 파일 복사됨)

🎉 Ready to Rock! 모든 셋업과 모듈 로드가 완벽히 끝났습니다.


In [2]:
# ==================================================================
# [Phase 6] Fast Tuning Test Harness (Demucs Bypassed)
# ==================================================================
import librosa
import numpy as np
import torch
import torchcrepe
import pretty_midi
import scipy.signal
from dataclasses import dataclass

# -----------------------------------------------------------------
# 1. Data Structure
# -----------------------------------------------------------------
@dataclass(frozen=True)
class NoteEvent:
    time: float
    duration: float
    midi_note: int
    confidence: float

# -----------------------------------------------------------------
# 2. Tracking Module (Type A Error Tuning Target)
# -----------------------------------------------------------------
def get_f0_crepe_robust_tuned(audio, sr, hop_length=160, fmax=500):
    """
    [Tuning Target 1] CREPE Pitch Tracking & Dynamic Thresholding
    """
    sos = scipy.signal.butter(4, 35, 'hp', fs=sr, output='sos')
    audio = scipy.signal.sosfilt(sos, audio).astype(np.float32)
    audio = librosa.util.normalize(audio)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # 추론 (메모리 문제 방지를 위해 chunking 로직 축약 - 단일 테스트용)
    audio_tensor = torch.tensor(audio).unsqueeze(0).to(device)
    f0, confidence = torchcrepe.predict(
        audio_tensor, sr, hop_length=hop_length, fmin=40, fmax=fmax,
        model='tiny', decoder=torchcrepe.decode.argmax,
        return_periodicity=True, device=device
    )

    f0 = f0.squeeze().cpu().numpy()
    confidence = confidence.squeeze().cpu().numpy()

    # ---------------------------------------------------------
    # 🔴 [HYPERPARAMETERS TO TUNE] 🔴
    # ---------------------------------------------------------
    # 1. Base Noise Gate (저음역대 포함 전체 적용)
    BASE_CONF_THRESH = 0.4

    # 2. High-Frequency Gate (타악기 블리딩 및 배음 에러 방어선)
    HIGH_FREQ_HZ = 200.0
    HIGH_FREQ_CONF_THRESH = 0.7
    # ---------------------------------------------------------

    # Apply Base Threshold
    f0[confidence < BASE_CONF_THRESH] = np.nan

    # Apply Dynamic Threshold (고주파 대역 엄격한 필터링)
    high_freq_mask = (f0 > HIGH_FREQ_HZ) & (confidence < HIGH_FREQ_CONF_THRESH)
    f0[high_freq_mask] = np.nan

    # Apply Physical Upper Bound
    f0[f0 > fmax] = np.nan

    return f0, confidence

# -----------------------------------------------------------------
# 3. Parsing Module (Type B Error Tuning Target)
# -----------------------------------------------------------------
def parse_f0_to_events_tuned(f0_array, confidence_array, hop_length=160, sr=16000):
    """
    [Tuning Target 2] Debouncing & Note Extraction
    """
    events = []
    frame_time = hop_length / sr
    current_note = None
    note_start_frame = 0
    blank_counter = 0

    # ---------------------------------------------------------
    # 🔴 [HYPERPARAMETERS TO TUNE] 🔴
    # ---------------------------------------------------------
    # 1. 최소 유지 프레임 (5 = 50ms). 이보다 짧은 음은 노이즈로 간주.
    MIN_DURATION_FRAMES = 5

    # 2. 결측치 관용도 (5 = 50ms). 이 프레임 내의 NaN은 같은 음표로 이어붙임.
    TOLERANCE_FRAMES = 5
    # ---------------------------------------------------------

    valid_mask = (f0_array > 0) & (~np.isnan(f0_array))
    midi_array = np.full(len(f0_array), np.nan)
    midi_array[valid_mask] = np.round(librosa.hz_to_midi(f0_array[valid_mask]))

    for i, midi_val in enumerate(midi_array):
        if not np.isnan(midi_val):
            midi_note = int(midi_val)
            blank_counter = 0
            if current_note is None:
                current_note = midi_note
                note_start_frame = i
            elif current_note != midi_note:
                duration_frames = i - note_start_frame
                if duration_frames >= MIN_DURATION_FRAMES:
                    conf = float(np.mean(confidence_array[note_start_frame:i]))
                    events.append(NoteEvent(note_start_frame * frame_time, duration_frames * frame_time, current_note, conf))
                current_note = midi_note
                note_start_frame = i
        else:
            blank_counter += 1
            if current_note is not None and blank_counter >= TOLERANCE_FRAMES:
                end_idx = i - blank_counter
                duration_frames = end_idx - note_start_frame
                if duration_frames >= MIN_DURATION_FRAMES:
                    conf = float(np.mean(confidence_array[note_start_frame:end_idx]))
                    events.append(NoteEvent(note_start_frame * frame_time, duration_frames * frame_time, current_note, conf))
                current_note = None

    return events

# -----------------------------------------------------------------
# 4. MIDI Export & Runner
# -----------------------------------------------------------------
def events_to_midi(events, output_path="tuned_output.mid"):
    midi = pretty_midi.PrettyMIDI()
    bass_program = pretty_midi.instrument_name_to_program('Electric Bass (finger)')
    bass_track = pretty_midi.Instrument(program=bass_program)

    for event in events:
        # Confidence를 64~127 사이의 Velocity로 스케일링
        velocity = int(64 + (event.confidence * 63))
        note = pretty_midi.Note(
            velocity=velocity,
            pitch=event.midi_note,
            start=event.time,
            end=event.time + event.duration
        )
        bass_track.notes.append(note)

    midi.instruments.append(bass_track)
    midi.write(output_path)
    print(f"✅ MIDI 추출 완료: {output_path} (총 {len(events)}개 노트)")

def run_tuning_test(audio_path):
    print(f"🎵 오디오 로딩 중: {audio_path}")
    y, sr = librosa.load(audio_path, sr=16000, mono=True)

    print("🧠 피치 트래킹 수행 중 (CREPE)...")
    f0, conf = get_f0_crepe_robust_tuned(y, sr)

    print("⚙️ 이벤트 파싱 및 디바운싱 수행 중...")
    events = parse_f0_to_events_tuned(f0, conf)

    out_midi_path = "/content/tuned_test_result.mid"
    events_to_midi(events, out_midi_path)
    return out_midi_path

In [3]:
# ==================================================================
# 🚀 실행부 (경로를 실제 구글 드라이브 테스트 파일 경로로 수정하세요)
# ==================================================================
TEST_AUDIO_PATH = "/content/drive/MyDrive/Bass_separator/dataset/performance_test_demo(bass).wav"
result_midi = run_tuning_test(TEST_AUDIO_PATH)

🎵 오디오 로딩 중: /content/drive/MyDrive/Bass_separator/dataset/performance_test_demo(bass).wav
🧠 피치 트래킹 수행 중 (CREPE)...
⚙️ 이벤트 파싱 및 디바운싱 수행 중...
✅ MIDI 추출 완료: /content/tuned_test_result.mid (총 207개 노트)


In [4]:
import shutil

# Source path is the result_midi generated earlier
source_path = result_midi

# Destination path in Google Drive
destination_dir = "/content/drive/MyDrive/Bass_separator/"
destination_file_name = "midi_tunning_test.mid"
destination_path = destination_dir + destination_file_name

# Ensure the destination directory exists
import os
os.makedirs(destination_dir, exist_ok=True)

# Copy the file
try:
    shutil.copy(source_path, destination_path)
    print(f"✅ MIDI 파일이 Google Drive에 성공적으로 저장되었습니다: {destination_path}")
except Exception as e:
    print(f"❌ MIDI 파일 저장 중 오류 발생: {e}")

✅ MIDI 파일이 Google Drive에 성공적으로 저장되었습니다: /content/drive/MyDrive/Bass_separator/midi_tunning_test.mid
